### Set Up

In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"

In [2]:
import keras
import torch
import numpy as np
import pandas as pd
from PIL import Image
from torchvision import transforms

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

keras.mixed_precision.set_global_policy("mixed_float16")
keras.config.set_image_data_format("channels_first")

print("Device:", device)

Device: cuda


### Load Model

In [4]:
model_name = "model/alexnet.keras"

In [5]:
@keras.saving.register_keras_serializable()
class LocalResponseNormalization(keras.layers.Layer):
    def call(self, x):
        return torch.nn.functional.local_response_norm(x, size=5, alpha=1e-4, beta=0.75, k=2.0, )

In [6]:
model = keras.models.load_model(model_name)

### Utils

In [7]:
class_map_file = "../../Datasets/ILSVRC2010_images/IDX_WNID_CLASS.csv"
class_map = pd.read_csv(class_map_file)
idx_to_wnid = dict(zip(class_map["idx"], class_map["wnid"]))
wnid_to_class = dict(zip(class_map["wnid"], class_map["class_name"]))

In [8]:
def get_class_name(target_id):
    return wnid_to_class[idx_to_wnid[target_id]]

In [9]:
def get_class_name(target_id):
    return wnid_to_class[idx_to_wnid[target_id]]

In [10]:
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

### Load Image

In [11]:
image_file = "test_image.JPEG"

In [12]:
image = Image.open(image_file).convert("RGB")
x = preprocess(image)
x = x.unsqueeze(0)
x = x.to(device)

### inference

In [13]:
with torch.inference_mode():
    predictions = model(x)
    predictions = predictions.float()
    predictions = predictions.cpu().numpy()

In [14]:
predicted_class = get_class_name(np.argmax(predictions[0]))
predicted_class

'airliner'

In [17]:
top5 = np.argsort(predictions[0])[-5:][::-1]

for idx in top5:

    print(
        f"{get_class_name(idx):30s} | "
        f"{100 * predictions[0][idx]:.2f}%"
    )

airliner                       | 84.96%
warplane, military plane       | 10.30%
airship, dirigible             | 2.55%
missile                        | 0.54%
submarine, pigboat, sub, U-boat | 0.51%


end